# MinimaxRegret: Minimize Your Maximum Code Regrets

This notebook demonstrates a transformer-based model that predicts potential errors in Python code before you run it. It combines static analysis with machine learning to identify issues and suggest fixes.

## Key Features:

1. **Pre-execution Error Detection**: Finds potential bugs without running the code
2. **Transformer Architecture**: Uses a small transformer model similar to nanoGPT
3. **Multi-level Analysis**: Combines static pattern matching with ML-based prediction
4. **Attention Visualization**: Shows what parts of the code the model focuses on
5. **Educational Feedback**: Provides slightly condescending but helpful error messages

In [4]:
import sys
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import difflib
from IPython.display import display, HTML

# Add the parent directory to the path if needed
sys.path.append('../src')

# Import our model and dataset
from model import MinimaxRegretModel
from dataset import CodeErrorDataset
from analysis import MinimaxRegret
from training import train_model, evaluate_model

## Initialize the Model and Dataset

First, let's load our pre-trained model and dataset.

In [10]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Create dataset
dataset = CodeErrorDataset()

# Create model
model = MinimaxRegretModel(
    vocab_size=dataset.vocab_size,
    n_embd=128,       # Embedding dimension
    n_head=4,         # Number of attention heads
    n_layer=3,        # Number of transformer layers
    block_size=512,   # Maximum sequence length
    dropout=0.1
).to(device)

# Check if model file exists, otherwise train
model_path = 'minimax_regret_model.pt'
if os.path.exists(model_path):
    print(f"Loading model from {model_path}")
    model.load_state_dict(torch.load(model_path, map_location=device))
else:
    print("Training new model (this is quick due to tiny model size)")
    train_model(model, dataset, batch_size=4, epochs=1)
    torch.save(model.state_dict(), model_path)

# Create MinimaxRegret instance
regret = MinimaxRegret(model, dataset)

# Print model info
print(f"Model parameters: {model.get_parameter_count():,}")
print(f"Dataset examples: {len(dataset)}")
print(f"Error types: {', '.join(dataset.error_types)}")


Using device: cpu
Dataset loaded with 10 examples and vocabulary size 75
Error types: IndexError, KeyError, ModuleNotFoundError, NameError, RecursionError, SyntaxError, TypeError, ZeroDivisionError
Model initialized with 746,069 parameters
Training new model (this is quick due to tiny model size)


RuntimeError: stack expects each tensor to be equal size, but got [97] at entry 0 and [30] at entry 1

## Interactive Code Analysis

Let's create a function to analyze code and display the results in a user-friendly way.

In [ ]:
def highlight_code_differences(original_code, fixed_code):
    """Create HTML highlighting the differences between original and fixed code"""
    original_lines = original_code.split('\n')
    fixed_lines = fixed_code.split('\n')
    d = difflib.Differ()
    diff = list(d.compare(original_lines, fixed_lines))
    html = ['<div style="font-family: monospace; white-space: pre;">\n']
    for line in diff:
        if line.startswith('+ '):
            html.append(f'<span style="background-color: #ccffcc">{line[2:]}</span>')
        elif line.startswith('- '):
            html.append(f'<span style="background-color: #ffcccc">{line[2:]}</span>')
        elif line.startswith('  '):
            html.append(line[2:])
    html.append('</div>')
    return '\n'.join(html)

def analyze_code(code, show_attention=False):
    """Analyze code and display the results"""
    results = regret.analyze(code)
    code_html = f"<pre style='background-color: #f5f5f5; padding: 10px; border-radius: 5px;'><code>{code}</code></pre>"
    display(HTML(code_html))
    if not results:
        display(HTML("<div style='color: green; font-weight: bold;'>No issues detected! (But that doesn't mean your code works...)</div>"))
        return
    table_html = """
    <table style='width: 100%; border-collapse: collapse;'>    
    <tr style='background-color: #4CAF50; color: white;'>
        <th style='padding: 8px; text-align: left;'>Line</th>
        <th style='padding: 8px; text-align: left;'>Error Type</th>
        <th style='padding: 8px; text-align: left;'>Description</th>
        <th style='padding: 8px; text-align: left;'>Suggestion</th>
    </tr>
    """
    for i, result in enumerate(results):
        bg_color = '#f2f2f2' if i % 2 == 0 else 'white'
        suggestion = result['suggestion'].replace('\n', '<br>')
        table_html += f"""
        <tr style='background-color: {bg_color};'>
            <td style='padding: 8px;'>{result['line']}:{result.get('col', 0)}</td>
            <td style='padding: 8px;'>{result['error_type']}</td>
            <td style='padding: 8px;'>{result['description']}</td>
            <td style='padding: 8px;'>{suggestion}</td>
        </tr>
        """
    table_html += "</table>"
    display(HTML(f"<div style='color: red; font-weight: bold;'>Found {len(results)} potential issues:</div>"))
    display(HTML(table_html))
    if show_attention and 'attention_weights' in results[0]:
        plt = regret.visualize_attention(code, results[0]['attention_weights'])
        plt.show()
    ml_results = [r for r in results if 'suggestion' in r and r['suggestion'].startswith('Consider this fixed version')]  
    if ml_results:
        fixed_code = ml_results[0]['suggestion'].replace('Consider this fixed version:\n', '')
        display(HTML("<h3>Code Comparison (Original vs. Fixed)</h3>"))
        display(HTML(highlight_code_differences(code, fixed_code)))

## Example 1: Division by Zero Error

In [ ]:
code_example_1 = """
def calculate_average(numbers):
    total = 0
    for num in numbers:
        total += num
    return total / len(numbers)
"""

analyze_code(code_example_1, show_attention=True)

## Example 2: Syntax Error

In [ ]:
code_example_2 = """
def greet(name):
    if name == 'Alice'
        return 'Hello, Alice!'
    else:
        return 'Hello, stranger!'
"""

analyze_code(code_example_2)

## Example 3: Index Error

In [ ]:
code_example_3 = """
def get_last_item(items):
    return items[len(items)]
"""

analyze_code(code_example_3)

## Example 4: Type Error

In [ ]:
code_example_4 = """
x = 10
y = '20'
result = x + y
"""

analyze_code(code_example_4)

## Example 5: Name Error

In [ ]:
code_example_5 = """
def process_data():
    data = get_data()
    return analyze(data)
"""

analyze_code(code_example_5)

## Try Your Own Code

Enter your own Python code to analyze:

In [ ]:
your_code = """
# Enter your Python code here
"""

analyze_code(your_code, show_attention=True)

## Model Evaluation

Let's evaluate the model's performance on the dataset:

In [ ]:
metrics = evaluate_model(model, dataset)

print(f"Evaluation Results:")
print(f"Loss: {metrics['loss']:.4f}")
print(f"Error Type Accuracy: {metrics['error_type_accuracy']:.2%}")
print(f"Error Location Accuracy: {metrics['error_loc_accuracy']:.2%}")

plt.figure(figsize=(10, 5))
plt.bar(['Error Type Accuracy', 'Error Location Accuracy'], 
        [metrics['error_type_accuracy'], metrics['error_loc_accuracy']], 
        color=['#4CAF50', '#2196F3'])
plt.title('Model Accuracy Metrics')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.axhline(y=0.5, color='r', linestyle='-', alpha=0.3)
plt.grid(axis='y', alpha=0.3)

for i, v in enumerate([metrics['error_type_accuracy'], metrics['error_loc_accuracy']]):
    plt.text(i, v + 0.02, f"{v:.2%}", ha='center')

plt.show()